# Train baseline YOLO26 on Roboflow tiles

The Roboflow YOLO detection export is in `data/Football Players and Ball.v6i.yolo26/` with its `data.yaml` and `train`, `valid`, `test` image/label folders. This notebook gathers the existing tiles, groups them by original frame, then resplits frame groups chronologically. The original export is untouched. It starts from pretrained `yolo26m.pt`, not your existing model.

The export filenames preserve source video and frame number but do not preserve tile positions. Each frame has 6 tiles or 18 images (three Roboflow variants per tile). The README documents exposure and blur augmentation. To avoid evaluating those variants, the notebook allocates frames chronologically, then retains only original `valid` and `test` export frames for evaluation. Consequently, some frames in the chronological validation and test ranges are excluded; the notebook prints counts. Request an unaugmented export for a complete unbiased chronological split. The tile hashes identify files, not grid positions. Inspect the grouping output before training.

Each tile needs a corresponding YOLO label file, including empty files for confirmed negatives. This export uses segmentation polygons; the notebook converts each polygon to its enclosing bounding box in the prepared copy for detection training. Install the project requirements and Jupyter in the notebook environment. The baseline weights may download on first use. Temporal splitting within one clip does not establish generalization to a new match; reserve separate footage for a stronger final test.


In [ ]:
from pathlib import Path
import json, math, re, shutil, hashlib
from collections import defaultdict
from datetime import datetime
import yaml, torch
from ultralytics import YOLO
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'requirements-yolo.txt').exists()), None)
if ROOT is None: raise RuntimeError('Start Jupyter from the repository or notebooks folder.')
DATASET_DIR = ROOT / 'data/Football Players and Ball.v6i.yolo26'
SOURCE_YAML = DATASET_DIR / 'data.yaml'
MODEL = 'yolo26m.pt'  # Standard pretrained baseline. Try yolo26s.pt for lower memory use.
TRAIN_FRACTION, VAL_FRACTION = 0.70, 0.15
FILENAME_PATTERN = r'(?P<sequence>.+)_frame_(?P<frame>\d+)_jpg\.rf\.(?P<tile>[0-9a-f]{32})'  # Roboflow hash identifies a file, not grid position.
EXPECTED_TILES_PER_FRAME = 6  # A train frame has 3 variants per tile (18 total).
IMAGE_SIZE = 960  # Existing tiles are resized/letterboxed by YOLO; try 1280 if validation supports it.
EPOCHS, BATCH, WORKERS, PATIENCE = 100, 2, 0, 25
DEVICE, ALLOW_CPU = 0, False  # Explicitly set DEVICE='cpu' and ALLOW_CPU=True to run on CPU.
CONFIDENCE, NMS_IOU, SEED = 0.10, 0.70, 42
print('Repository:', ROOT)


## 1. Inspect the export and hardware

The RX 6800 XT is absent from AMD's current official Radeon ROCm/PyTorch Windows and Linux support matrices. GPU training requires a compatible PyTorch installation; the notebook stops instead of silently running on CPU. ROCm uses PyTorch's `torch.cuda` interface. The YAML class names define your labels; the COCO baseline detection head will be adapted to these classes.


In [ ]:
print('PyTorch:', torch.__version__, '| HIP/ROCm:', torch.version.hip, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
if not SOURCE_YAML.is_file(): raise FileNotFoundError(f'Unzip the Roboflow export and set DATASET_DIR. Missing: {SOURCE_YAML}')
config = yaml.safe_load(SOURCE_YAML.read_text(encoding='utf-8'))
raw = config.get('names')
if isinstance(raw, list): names = raw
elif isinstance(raw, dict): names = [raw.get(i, raw.get(str(i))) for i in range(len(raw))]
else: raise ValueError('data.yaml must have a names list or integer-indexed mapping.')
if not names or any(not isinstance(n,str) or not n.strip() for n in names): raise ValueError('Invalid class names.')
images = sorted(p for p in DATASET_DIR.rglob('*') if p.is_file() and p.parent.name.lower() == 'images' and p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'})
print('Classes:', names, '| Images:', len(images))
for image in images[:20]: print(image.relative_to(DATASET_DIR))
if not images: raise ValueError('No images found in export images folders.')


## 2. Group, validate and split by source frame

The export contains one source sequence. Filenames contain frame numbers and unique hashes, not spatial tile positions; all hashes for a frame stay together. Frames are split chronologically before any evaluation filtering. The Roboflow README says the old training set has three augmented variants per source tile. We include these variants in chronological training, but **exclude old train-export frames from validation and test** because the hashes do not reliably identify unaugmented images. Only old `valid`/`test` frames contribute to evaluation. This yields fewer evaluation frames and can introduce selection bias; use a new unaugmented export to evaluate all chronological frames. Existing augmentation is baked into the training tiles, in addition to notebook training augmentation.


In [ ]:
if FILENAME_PATTERN is None: raise ValueError('Set FILENAME_PATTERN using the actual filenames; the notebook cannot infer it safely.')
pattern = re.compile(FILENAME_PATTERN)
if not {'frame','tile'}.issubset(pattern.groupindex): raise ValueError('Pattern needs named frame and tile groups.')
if not (0 < TRAIN_FRACTION < 1 and 0 < VAL_FRACTION < 1 and TRAIN_FRACTION + VAL_FRACTION < 1): raise ValueError('Invalid split fractions.')
groups, hashes, sequences = defaultdict(dict), {}, set()
def read_labels(image):
    label = image.parent.parent / 'labels' / (image.stem + '.txt')
    if not label.is_file(): raise FileNotFoundError(f'Missing label: {label}; create an empty file only for a confirmed negative.')
    boxes = []
    for number, line in enumerate(label.read_text(encoding='utf-8').splitlines(),1):
        try:
            values = list(map(float, line.split()))
            if not values or not all(math.isfinite(v) for v in values): raise ValueError('missing or non-finite value')
            cls, points = values[0], values[1:]
            if not cls.is_integer() or not 0 <= cls < len(names): raise ValueError('invalid class ID')
            if len(points) == 4:
                xc, yc, w, h = points
                if not (0 <= xc <= 1 and 0 <= yc <= 1 and 0 < w <= 1 and 0 < h <= 1 and xc-w/2 >= -1e-6 and xc+w/2 <= 1+1e-6 and yc-h/2 >= -1e-6 and yc+h/2 <= 1+1e-6): raise ValueError('invalid normalized box')
                boxes.append(f'{int(cls)} {xc:.8f} {yc:.8f} {w:.8f} {h:.8f}')
                continue
            if len(points) < 6 or len(points) % 2: raise ValueError('expected bounding box or polygon with at least 3 coordinate pairs')
            if any(v < 0 or v > 1 for v in points): raise ValueError('polygon coordinate outside tile')
            xs, ys = points[0::2], points[1::2]
            x1,x2,y1,y2 = min(xs),max(xs),min(ys),max(ys)
            if x2 <= x1 or y2 <= y1: raise ValueError('polygon has zero width or height')
            boxes.append(f"{int(cls)} {(x1+x2)/2:.8f} {(y1+y2)/2:.8f} {x2-x1:.8f} {y2-y1:.8f}")
        except ValueError as exc: raise ValueError(f'{label}:{number}: {exc}') from exc
    return boxes
for image in images:
    match = pattern.fullmatch(image.stem)
    if not match or not match.group('frame').isdigit() or not match.group('tile'): raise ValueError(f'Cannot identify frame/tile: {image.name}')
    sequences.add(match.group('sequence'))
    frame, tile = int(match.group('frame')), match.group('tile')
    if tile in groups[frame]: raise ValueError(f'Duplicate tile {tile} for frame {frame}: {image}')
    groups[frame][tile] = image
    read_labels(image)
    digest = hashlib.sha256(image.read_bytes()).hexdigest()
    if digest in hashes: raise ValueError(f'Duplicate image bytes: {image} and {hashes[digest]}; inspect export.')
    hashes[digest] = image
if len(sequences) != 1: raise ValueError(f'Expected one video sequence; found {sorted(sequences)}. Frame numbers alone cannot order multiple videos.')
for frame, tiles in groups.items():
    old_splits = {p.parent.parent.name.lower() for p in tiles.values()}
    if len(old_splits) != 1 or not old_splits <= {'train','valid','test'}: raise ValueError(f'Frame {frame} crosses unexpected export folders: {old_splits}')
    old_split = next(iter(old_splits))
    expected = EXPECTED_TILES_PER_FRAME * (3 if old_split == 'train' else 1)
    if len(tiles) != expected: raise ValueError(f'Frame {frame}: expected {expected} images in {old_split}, found {len(tiles)}.')
frames = sorted(groups); n = len(frames)
train_end, val_end = int(n*TRAIN_FRACTION), int(n*(TRAIN_FRACTION+VAL_FRACTION))
if not 0 < train_end < val_end < n: raise ValueError('Not enough frames for all three splits.')
splits = {'train':frames[:train_end], 'val':frames[train_end:val_end], 'test':frames[val_end:]}
for group, ids in splits.items():
    print(f'{group} chronological allocation: {len(ids)} source frames, {ids[0]}–{ids[-1]}')
allocated_splits = {name: ids[:] for name, ids in splits.items()}
for group in ('val', 'test'):
    splits[group] = [f for f in splits[group] if next(iter(groups[f].values())).parent.parent.name.lower() in {'valid', 'test'}]
    print(f'{group}: {len(splits[group])} clean frames retained; {len(allocated_splits[group])-len(splits[group])} pre-augmented frames excluded')
    if not splits[group]: raise ValueError(f'No unaugmented frames in chronological {group} range; obtain an unaugmented export.')
for group, ids in splits.items():
    print(f'{group}: {len(ids)} frames and {sum(len(groups[f]) for f in ids)} images used')
    for f in ids[:2]: print(' ', f, '→', [p.name for p in groups[f].values()][:3], '...')
print('Review grouping and order before copying.')


In [ ]:
# Copy into a new timestamped dataset; the original export is untouched.
output = ROOT / 'runs' / 'training-data' / datetime.now().strftime('%Y%m%d_%H%M%S_%f')
output.mkdir(parents=True, exist_ok=False)
manifest = {'chronological_allocations': allocated_splits, 'excluded_from_evaluation': {name: sorted(set(allocated_splits[name]) - set(splits[name])) for name in ('val', 'test')}}
for group, ids in splits.items():
    image_out, label_out = output/'images'/group, output/'labels'/group
    image_out.mkdir(parents=True); label_out.mkdir(parents=True)
    manifest[group] = []
    for frame in ids:
        for tile, image in sorted(groups[frame].items()):
            target = f'frame_{frame:012d}_tile_{len(manifest[group]):08d}'
            shutil.copy2(image, image_out/(target+image.suffix.lower()))
            boxes = read_labels(image)
            (label_out/(target+'.txt')).write_text('\n'.join(boxes) + ('\n' if boxes else ''), encoding='utf-8')
            manifest[group].append({'frame':frame,'tile':tile,'source':str(image.relative_to(DATASET_DIR)),'output':target})
    print(group, ': copied', len(manifest[group]), 'tiles')
(output/'split_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
dataset_yaml = output/'dataset.yaml'
dataset_yaml.write_text(yaml.safe_dump({'path':str(output.resolve()),'train':'images/train','val':'images/val','test':'images/test','names':names},sort_keys=False),encoding='utf-8')
print('Prepared:', dataset_yaml)


## 3. Train the baseline

Ultralytics adapts the pretrained model's head to your dataset classes. The Roboflow training variants already include exposure and blur changes. Additional mild color, translation, scale and horizontal-flip augmentations apply only to training; mosaic and mixup start disabled. YOLO letterboxes and normalizes existing tiles. Start with medium, size 960 and batch 2. Try batch 1 or YOLO26s if memory or speed is a problem. The current frontend uses whole-frame inference, not tiled inference, so its results may differ from tile-level evaluation.


In [ ]:
if DEVICE != 'cpu' and not torch.cuda.is_available(): raise RuntimeError('No PyTorch GPU detected. Set up a compatible stack or explicitly choose CPU.')
if DEVICE == 'cpu' and not ALLOW_CPU: raise RuntimeError('CPU training requires ALLOW_CPU=True.')
model = YOLO(MODEL)  # Standard YOLO26 baseline; pretrained weights may download on first use.
results = model.train(data=str(dataset_yaml), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH,
    device=DEVICE, workers=WORKERS, patience=PATIENCE, seed=SEED,
    project=str(ROOT/'runs'/'local-training'), name='football', exist_ok=False,
    fliplr=0.5, flipud=0.0, hsv_h=0.01, hsv_s=0.25, hsv_v=0.20,
    degrees=0.0, translate=0.05, scale=0.15, mosaic=0.0, mixup=0.0)
best = Path(results.save_dir)/'weights'/'best.pt'
if not best.is_file(): raise FileNotFoundError(f'No best checkpoint found: {best}')
print('Best checkpoint:', best)


## 4. Evaluate and inspect

Validation is for choosing confidence, NMS IoU and model settings. `iou` here is the overlap threshold for non-maximum suppression, **not** the IoU used to report mAP@50 or mAP@50–95. Higher NMS IoU retains more overlapping predictions, potentially increasing duplicates; confidence often has a more direct effect on recall. YOLO26 may use an end-to-end NMS-free inference path, in which case changing NMS IoU has no effect. Check your installed Ultralytics version and prediction mode. Test is held out until you finish tuning. Inspect per-class ball recall, precision and confusion matrix, not just overall mAP. Tile-level metrics can count an object more than once when it crosses a tile boundary.


In [ ]:
best_model = YOLO(str(best))
val_metrics = best_model.val(data=str(dataset_yaml), split='val', imgsz=IMAGE_SIZE, batch=BATCH,
                             device=DEVICE, workers=WORKERS, conf=CONFIDENCE, iou=NMS_IOU,
                             project=str(ROOT / 'runs' / 'local-training'), name='validation', exist_ok=False)
print('Validation mAP50:', val_metrics.box.map50, 'mAP50-95:', val_metrics.box.map)
print('Per-class AP50:', val_metrics.box.ap50)


In [ ]:
# Run only after deciding settings from validation. This reports tile-level metrics.
test_metrics = best_model.val(data=str(dataset_yaml), split='test', imgsz=IMAGE_SIZE, batch=BATCH,
                              device=DEVICE, workers=WORKERS, conf=CONFIDENCE, iou=NMS_IOU,
                              project=str(ROOT / 'runs' / 'local-training'), name='test', exist_ok=False)
print('Test mAP50:', test_metrics.box.map50, 'mAP50-95:', test_metrics.box.map)
print('Per-class AP50:', test_metrics.box.ap50)
print('Checkpoint to integrate after review:', best)


Review `results.csv`, curves and confusion matrices in the run directory. Keep the split manifest and document the baseline, hardware, settings and results. This notebook does not replace the frontend model automatically.
